In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(
    "../data/raw/career_dataset.csv"
)

print(
    "Dataset shape:",
    df.shape
)

Dataset shape: (10000, 46)


In [3]:
role_requirements = {
    "Data Analyst": {
        "python_score": 0.15,
        "sql_score": 0.25,
        "excel_score": 0.20,
        "powerbi_score": 0.20,
        "statistics_score": 0.10,
        "data_visualization_score": 0.10
    },

    "Data Scientist": {
        "python_score": 0.25,
        "sql_score": 0.10,
        "statistics_score": 0.20,
        "ml_score": 0.25,
        "coding_score": 0.10,
        "data_visualization_score": 0.10
    },

    "Business Analyst": {
        "sql_score": 0.15,
        "excel_score": 0.25,
        "powerbi_score": 0.15,
        "statistics_score": 0.10,
        "communication_score": 0.20,
        "presentation_score": 0.15
    },

    "BI Analyst": {
        "sql_score": 0.25,
        "excel_score": 0.20,
        "powerbi_score": 0.25,
        "data_visualization_score": 0.15,
        "statistics_score": 0.10,
        "communication_score": 0.05
    },

    "Software Developer": {
        "python_score": 0.15,
        "java_score": 0.25,
        "coding_score": 0.30,
        "web_dev_score": 0.15,
        "logical_reasoning_score": 0.15
    },

    "ML Engineer": {
        "python_score": 0.25,
        "ml_score": 0.30,
        "coding_score": 0.20,
        "statistics_score": 0.10,
        "sql_score": 0.10,
        "logical_reasoning_score": 0.05
    }
}

In [4]:
def calculate_role_match(
    student,
    role_requirements
):
    
    scores = {}

    for role, requirements in role_requirements.items():

        total_score = 0

        for skill, weight in requirements.items():

            student_score = student[skill]

            normalized_score = student_score / 100

            total_score += (
                normalized_score * weight
            )

        scores[role] = round(
            total_score * 100,
            2
        )

    return scores

In [5]:
student = df.iloc[0]

In [6]:
role_scores = calculate_role_match(
    student,
    role_requirements
)

print(role_scores)

{'Data Analyst': np.float64(58.51), 'Data Scientist': np.float64(60.41), 'Business Analyst': np.float64(60.58), 'BI Analyst': np.float64(56.05), 'Software Developer': np.float64(54.77), 'ML Engineer': np.float64(58.04)}


In [7]:
recommendations = pd.DataFrame(
    role_scores.items(),
    columns=[
        "Role",
        "Match_Score"
    ]
)

In [8]:
recommendations = recommendations.sort_values(
    "Match_Score",
    ascending=False
)

In [9]:
recommendations

,Role,Match_Score
2,Business Analyst,60.58
1,Data Scientist,60.41
0,Data Analyst,58.51
5,ML Engineer,58.04
3,BI Analyst,56.05
4,Software Developer,54.77


In [10]:
top_roles = recommendations.head(3)

print(top_roles)

               Role  Match_Score
2  Business Analyst        60.58
1    Data Scientist        60.41
0      Data Analyst        58.51


In [11]:
def skill_status(score):

    if score >= 75:
        return "Strong"

    elif score >= 50:
        return "Needs Improvement"

    else:
        return "Skill Gap"

In [12]:
def get_skill_gap(
    student,
    role
):

    requirements = role_requirements[role]

    results = []

    for skill, weight in requirements.items():

        score = student[skill]

        results.append({
            "Skill": skill,
            "Score": round(score, 2),
            "Status": skill_status(score),
            "Importance": weight
        })

    return pd.DataFrame(results)

In [13]:
top_role = top_roles.iloc[0]["Role"]

print(
    "Recommended Role:",
    top_role
)

Recommended Role: Business Analyst


In [14]:
skill_gap = get_skill_gap(
    student,
    top_role
)

skill_gap

,Skill,Score,Status,Importance
0,sql_score,97.70,Strong,0.15
1,excel_score,39.85,Skill Gap,0.25
2,powerbi_score,23.07,Skill Gap,0.15
3,statistics_score,68.04,Needs Improvement,0.10
4,communication_score,64.57,Needs Improvement,0.20
5,presentation_score,85.22,Strong,0.15


In [15]:
skill_gap["Gap"] = (
    100 - skill_gap["Score"]
)

In [16]:
skill_gap["Priority"] = (
    skill_gap["Gap"]
    * skill_gap["Importance"]
)

In [17]:
skill_gap["Priority"] = (
    skill_gap["Gap"]
    * skill_gap["Importance"]
)

In [18]:
learning_resources = {
    "python_score": "Advanced Python",
    "sql_score": "Advanced SQL",
    "excel_score": "Advanced Excel",
    "powerbi_score": "Power BI & DAX",
    "statistics_score": "Statistics for Data Analytics",
    "ml_score": "Machine Learning",
    "coding_score": "Data Structures & Algorithms",
    "data_visualization_score": "Data Visualization",
    "communication_score": "Business Communication",
    "presentation_score": "Presentation & Storytelling",
    "logical_reasoning_score": "Logical Reasoning"
}

In [20]:
# Get skill gaps for the selected top role

top_role = top_roles.iloc[0]["Role"]

print("Recommended Role:", top_role)

skill_gap = get_skill_gap(
    student,
    top_role
)

# Calculate skill gap
skill_gap["Gap"] = 100 - skill_gap["Score"]

# Calculate priority
skill_gap["Priority"] = (
    skill_gap["Gap"] * skill_gap["Importance"]
)

# Sort by priority
priority_gaps = skill_gap.sort_values(
    "Priority",
    ascending=False
).reset_index(drop=True)

print("\nSkill Gap Analysis:")
display(priority_gaps)

Recommended Role: Business Analyst

Skill Gap Analysis:


,Skill,Score,Status,Importance,Gap,Priority
0,excel_score,39.85,Skill Gap,0.25,60.15,15.0375
1,powerbi_score,23.07,Skill Gap,0.15,76.93,11.5395
2,communication_score,64.57,Needs Improvement,0.20,35.43,7.0860
3,statistics_score,68.04,Needs Improvement,0.10,31.96,3.1960
4,presentation_score,85.22,Strong,0.15,14.78,2.2170
5,sql_score,97.70,Strong,0.15,2.30,0.3450


In [21]:
roadmap = []

for _, row in priority_gaps.iterrows():

    skill = row["Skill"]

    if row["Status"] != "Strong":

        roadmap.append({
            "Skill": skill,
            "Priority": round(
                row["Priority"],
                2
            ),
            "Recommended_Learning":
                learning_resources.get(
                    skill,
                    skill
                )
        })

In [22]:
roadmap_df = pd.DataFrame(
    roadmap
)

roadmap_df

,Skill,Priority,Recommended_Learning
0,excel_score,15.04,Advanced Excel
1,powerbi_score,11.54,Power BI & DAX
2,communication_score,7.09,Business Communication
3,statistics_score,3.20,Statistics for Data Analytics


In [23]:
all_recommendations = []

for index, student in df.iterrows():

    scores = calculate_role_match(
        student,
        role_requirements
    )

    best_role = max(
        scores,
        key=scores.get
    )

    all_recommendations.append({
        "student_id": student["student_id"],
        "recommended_role": best_role,
        "match_score": scores[best_role]
    })

In [24]:
recommendation_df = pd.DataFrame(
    all_recommendations
)

recommendation_df.head()

,student_id,recommended_role,match_score
0,10001,Business Analyst,60.58
1,10002,Business Analyst,55.55
2,10003,Data Scientist,66.14
3,10004,ML Engineer,80.62
4,10005,Business Analyst,66.72


In [25]:
recommendation_df.to_csv(
    "../reports/career_recommendations.csv",
    index=False
)

print(
    "Career recommendations saved!"
)

Career recommendations saved!
